# AI-09 前半 — Integration reference export
既存のAI-03/05/08成果物を再利用します。学習・統計の再計算は行いません。
各見出し単位で上から実行してください。既存 `src/preprocess.py` / `src/dataset.py` は変更しません。
TESTから各クラス3件をSHA-256順で選定し、初回CSVを固定します。正解した例だけを選ぶ処理はありません。
この9件は統合確認用であり、モデル精度評価の代替ではありません。


## 1. Drive接続（Colab）

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. パス・環境確認
AI-08と同じ依存バージョンを使用してください。パッケージの自動更新は行いません。

In [2]:
from pathlib import Path
import sys, json, hashlib, platform, subprocess, shutil
import numpy as np
import pandas as pd
import scipy
import tensorflow as tf
from scipy.io import wavfile

AI_ROOT = Path('/content/drive/MyDrive/TRON/AI')
DATASET_ROOT = Path('/content/drive/MyDrive/TRON/AI/dataset')
CLIP_ROOT = DATASET_ROOT / 'processed/clips_16k_1s'
STATS = DATASET_ROOT / 'metadata/05_split_info/normalization_stats.npz'
SPLITS = DATASET_ROOT / 'metadata/05_split_info/split_manifest.csv'
CLIPS = DATASET_ROOT / 'metadata/03_clip_info/clip_manifest.csv'
MODEL = AI_ROOT / 'model/baseline_int8.tflite'
OUT = DATASET_ROOT / 'metadata/09_integration_info'
GENERATED = AI_ROOT / 'generated'
for p in [STATS, SPLITS, CLIPS, MODEL, AI_ROOT/'src/preprocess.py', AI_ROOT/'src/dataset.py', AI_ROOT/'src/export_integration.py']:
    assert p.is_file(), f'Missing: {p}'
sys.path.insert(0, str(AI_ROOT))
from src import preprocess as pp
from src.dataset import load_normalization_stats, load_split_manifest, select_split, load_feature
from src.export_integration import quantize_int8, dequantize_int8, model_quantization, sha256_file
assert Path(pp.__file__).resolve() == (AI_ROOT/'src/preprocess.py').resolve()
OUT.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__, 'NumPy:', np.__version__, 'SciPy:', scipy.__version__)

TensorFlow: 2.20.0 NumPy: 2.1.3 SciPy: 1.16.3


## 3. AI-05統計とAI-03/05 manifestの整合性

In [3]:
mean, std = load_normalization_stats(STATS)
assert mean.shape == std.shape == (40,)
assert np.isfinite(mean).all() and np.isfinite(std).all() and (std >= pp.STD_EPS).all()
manifest = load_split_manifest(SPLITS)
clips = pd.read_csv(CLIPS)
keys = ['clip_id', 'relative_path', 'class_id', 'class_name']
for df in (manifest, clips):
    assert set(keys).issubset(df.columns)
    assert df[keys].notna().all().all()
    assert df.clip_id.is_unique and df.relative_path.is_unique
    assert all({0:'OTHER',1:'SNORE',2:'COUGH'}.get(i) == n for i,n in zip(df.class_id, df.class_name))
check = manifest[keys].merge(clips[keys], on='clip_id', suffixes=('_05','_03'), validate='one_to_one', how='left')
for k in keys[1:]:
    assert (check[k+'_05'] == check[k+'_03']).all(), f'AI-03/05 mismatch: {k}'
test_df = select_split(manifest, 'TEST')
assert len(test_df) > 0
print(test_df.groupby('class_name').size())
# TRAIN-only provenance follows the AI-05 artifact; do not estimate stats here.

class_name
COUGH    768
OTHER    228
SNORE    166
dtype: int64


## 4. モデル入出力と前処理仕様の固定確認

In [4]:
interpreter = tf.lite.Interpreter(model_path=str(MODEL), num_threads=1)
interpreter.allocate_tensors()
in_detail, out_detail = model_quantization(interpreter)
assert (pp.SAMPLE_RATE, pp.INPUT_SAMPLES, pp.FRAME_LENGTH, pp.FRAME_STEP, pp.FFT_LENGTH) == (16000,16000,400,160,512)
assert (pp.NUM_FRAMES, pp.NUM_MEL_BINS, pp.FMIN, pp.FMAX) == (98,40,60,8000)
assert pp.MEL_FILTERBANK.shape == (257,40)
np.testing.assert_allclose(pp.periodic_hann_window(), 0.5-0.5*np.cos(2*np.pi*np.arange(400)/400), atol=6e-8, rtol=0)
np.testing.assert_allclose(pp.preprocess_pcm16(np.zeros(16000, np.int16)), -100, atol=1e-5)
assert pp.LOG_FLOOR == 1e-10
# Python np.rint: ties-to-even, zero point is added AFTER rounding.
np.testing.assert_array_equal(quantize_int8(np.array([-2.5,-1.5,-0.5,0.5,1.5,2.5]), 1, 2), [0,0,2,2,4,4])
print(in_detail['quantization'], out_detail['quantization'])

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


(0.021447716280817986, 2) (0.00390625, -128)


## 5. 固定ベクトル選定
既存CSVがあれば再利用し、所属TEST・ラベル・WAVハッシュを再検証します。変更時は旧成果物一式を別フォルダへ退避してから、新しい出力先で実行してください。

In [5]:
VECTOR_CSV = OUT / 'integration_test_vectors.csv'
CLASSES = {0:'OTHER', 1:'SNORE', 2:'COUGH'}
SEED = 2026

def clip_path(relative):
    rel = Path(str(relative))
    assert not rel.is_absolute() and '..' not in rel.parts
    path = (CLIP_ROOT / rel).resolve()
    assert path.is_relative_to(CLIP_ROOT.resolve()) and path.is_file()
    return path

if VECTOR_CSV.exists():
    vectors = pd.read_csv(VECTOR_CSV, keep_default_na=False)
else:
    selected = []
    for class_id, class_name in CLASSES.items():
        candidates = test_df[test_df.class_id == class_id].copy()
        assert len(candidates) >= 3
        candidates['selection_key'] = candidates.clip_id.map(lambda x: hashlib.sha256(f'{SEED}:{x}'.encode()).hexdigest())
        selected.append(candidates.sort_values(['selection_key','clip_id']).head(3))
    vectors = pd.concat(selected, ignore_index=True)
    vectors.insert(0, 'vector_id', [f'v{i:02d}' for i in range(len(vectors))])
    vectors['wav_sha256'] = [sha256_file(clip_path(p)) for p in vectors.relative_path]
assert len(vectors) == 9 and vectors.vector_id.is_unique and vectors.clip_id.is_unique
assert vectors.vector_id.tolist() == [f'v{i:02d}' for i in range(9)]
assert vectors.groupby('class_id').size().to_dict() == {0:3,1:3,2:3}
for row in vectors.itertuples(index=False):
    match = test_df[test_df.clip_id == row.clip_id]
    assert len(match) == 1
    for k in ['relative_path','class_id','class_name','split']:
        assert getattr(row,k) == match.iloc[0][k]
    assert sha256_file(clip_path(row.relative_path)) == row.wav_sha256
vectors.to_csv(VECTOR_CSV, index=False)
display(vectors[['vector_id','clip_id','class_name','relative_path']])

,vector_id,clip_id,class_name,relative_path
0,v00,esc50__3-251617-A-48,OTHER,OTHER/esc50__3-251617-A-48.wav
1,v01,esc50__3-150363-A-38,OTHER,OTHER/esc50__3-150363-A-38.wav
2,v02,esc50__4-181707-A-32,OTHER,OTHER/esc50__4-181707-A-32.wav
3,v03,audioset__audioset_jdCw86PiqMM_10000_event_000,SNORE,SNORE/audioset__audioset_jdCw86PiqMM_10000_eve...
4,v04,audioset__audioset_KAItJUveVr4_30000_event_000,SNORE,SNORE/audioset__audioset_KAItJUveVr4_30000_eve...
5,v05,audioset__audioset_DOzc8GAQq8A_20000_event_001,SNORE,SNORE/audioset__audioset_DOzc8GAQq8A_20000_eve...
6,v06,coswara__20200814_4ck5XZzCx0XBkREVkHGzpAi1aSk1...,COUGH,COUGH/coswara__20200814_4ck5XZzCx0XBkREVkHGzpA...
7,v07,coswara__20200901_REREbF9ZKUd33adQOFDQXSYK3SI3...,COUGH,COUGH/coswara__20200901_REREbF9ZKUd33adQOFDQXS...
8,v08,coswara__20200417_qr3tupBiFNgeVUX9N4ct8GigLP53...,COUGH,COUGH/coswara__20200417_qr3tupBiFNgeVUX9N4ct8G...


## 6. PC reference生成・TFLite推論
NPYはvector_id順のバッチ配列。PCM、未正規化Log-Mel、正規化特徴量、INT8入力、INT8出力、逆量子化出力を保存します。

In [6]:
buffers = {k:[] for k in ['pcm_int16','log_mel_float32','normalized_float32','input_int8','output_int8','output_dequantized_float32']}
rows = []
for row in vectors.itertuples(index=False):
    sr, pcm = wavfile.read(clip_path(row.relative_path))
    assert sr == 16000 and pcm.dtype == np.int16 and pcm.shape == (16000,)
    logmel = pp.preprocess_pcm16(pcm)
    normalized = pp.preprocess_pcm16(pcm, mean=mean, std=std)
    feature = load_feature(row.relative_path, clip_root=CLIP_ROOT, mean=mean, std=std)
    np.testing.assert_array_equal(feature[...,0], normalized)
    np.testing.assert_array_equal(normalized, pp.normalize_log_mel(logmel, mean, std))
    assert logmel.shape == normalized.shape == (98,40)
    assert logmel.dtype == normalized.dtype == np.float32
    assert np.isfinite(logmel).all() and np.isfinite(normalized).all()
    q = quantize_int8(feature[np.newaxis,...], *in_detail['quantization'])
    assert q.shape == (1,98,40,1) and q.dtype == np.int8
    interpreter.set_tensor(in_detail['index'], q)
    interpreter.invoke()
    y = interpreter.get_tensor(out_detail['index']).copy()
    assert y.shape == (1,3) and y.dtype == np.int8
    deq = dequantize_int8(y, *out_detail['quantization'])
    assert np.isfinite(deq).all()
    pred = int(y.argmax(axis=1)[0])
    assert pred == int(deq.argmax(axis=1)[0])
    for k,v in zip(buffers, [pcm,logmel,normalized,q[0],y[0],deq[0]]):
        buffers[k].append(v)
    record = dict(vector_id=row.vector_id, clip_id=row.clip_id, class_id=row.class_id,
                  class_name=row.class_name, predicted_class_id=pred, predicted_class_name=CLASSES[pred],
                  correct=(pred == row.class_id), saturation_fraction=float(np.mean((q == -128)|(q == 127))))
    for i,name in CLASSES.items():
        record[f'output_int8_{name}'] = int(y[0,i])
        record[f'output_dequantized_{name}'] = float(deq[0,i])
    rows.append(record)
arrays = {k:np.stack(v) for k,v in buffers.items()}
outputs = pd.DataFrame(rows)
display(outputs)

,vector_id,clip_id,class_id,class_name,predicted_class_id,predicted_class_name,correct,saturation_fraction,output_int8_OTHER,output_dequantized_OTHER,output_int8_SNORE,output_dequantized_SNORE,output_int8_COUGH,output_dequantized_COUGH
0,v00,esc50__3-251617-A-48,0.0,OTHER,0,OTHER,True,0.000000,115,0.949219,-116,0.046875,-127,0.003906
1,v01,esc50__3-150363-A-38,0.0,OTHER,0,OTHER,True,0.000000,94,0.867188,-102,0.101562,-120,0.031250
2,v02,esc50__4-181707-A-32,0.0,OTHER,0,OTHER,True,0.000000,127,0.996094,-128,0.000000,-128,0.000000
3,v03,audioset__audioset_jdCw86PiqMM_10000_event_000,1.0,SNORE,2,COUGH,False,0.019898,-121,0.027344,-69,0.230469,62,0.742188
4,v04,audioset__audioset_KAItJUveVr4_30000_event_000,1.0,SNORE,1,SNORE,True,0.000000,-106,0.085938,98,0.882812,-120,0.031250
5,v05,audioset__audioset_DOzc8GAQq8A_20000_event_001,1.0,SNORE,1,SNORE,True,0.000000,-98,0.117188,98,0.882812,-128,0.000000
6,v06,coswara__20200814_4ck5XZzCx0XBkREVkHGzpAi1aSk1...,2.0,COUGH,2,COUGH,True,0.009949,-128,0.000000,-128,0.000000,127,0.996094
7,v07,coswara__20200901_REREbF9ZKUd33adQOFDQXSYK3SI3...,2.0,COUGH,2,COUGH,True,0.001531,-128,0.000000,-128,0.000000,127,0.996094
8,v08,coswara__20200417_qr3tupBiFNgeVUX9N4ct8GigLP53...,2.0,COUGH,2,COUGH,True,0.000000,-126,0.007812,-128,0.000000,126,0.992188


## 7. Cヘッダー生成スクリプト呼び出し

In [7]:
subprocess.run([sys.executable, '-m', 'src.export_integration',
                '--stats', str(STATS), '--model', str(MODEL), '--output-dir', str(GENERATED)],
               cwd=AI_ROOT, check=True)
headers = ['normalization_params.h','hann_window.h','mel_filterbank.h','quantization_params.h']
assert all((GENERATED/p).is_file() for p in headers)

## 8. 保存・再読込検証・再現性情報
以前のreferenceと入力依存ファイルが異なる場合は上書きせず停止します。

In [8]:
run_manifest = {
    'schema_version': 1, 'selection_seed': SEED,
    'rounding': 'float32 division, ties-to-even before zero point, saturate int8',
    'normalization_source': 'AI-05 TRAIN-only mean/std (not recomputed)',
    'versions': {'python':platform.python_version(), 'numpy':np.__version__, 'scipy':scipy.__version__, 'tensorflow':tf.__version__},
    'source_sha256': {str(p):sha256_file(p) for p in [MODEL, STATS, SPLITS, CLIPS, AI_ROOT/'src/preprocess.py', AI_ROOT/'src/dataset.py', AI_ROOT/'src/export_integration.py']},
    'vector_ids': vectors.vector_id.tolist(),
    'wav_sha256': vectors.wav_sha256.tolist(),
    'arrays': {k:{'shape':list(v.shape),'dtype':str(v.dtype)} for k,v in arrays.items()},
    'header_sha256': {p:sha256_file(GENERATED/p) for p in headers},
}
run_path = OUT/'integration_run.json'
if run_path.exists():
    assert json.loads(run_path.read_text()) == run_manifest, 'Reference dependencies changed; use a new OUT directory.'
for name, a in arrays.items():
    path = OUT / (name + '.npy')
    if path.exists():
        np.testing.assert_array_equal(np.load(path, allow_pickle=False), a)
    np.save(path, a, allow_pickle=False)
    np.testing.assert_array_equal(np.load(path, allow_pickle=False), a)
outputs.to_csv(OUT/'pc_reference_outputs.csv', index=False, float_format='%.9g')
run_path.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
print('Saved:', OUT)
print('Headers:', GENERATED)

Saved: /content/drive/MyDrive/TRON/AI/dataset/metadata/09_integration_info
Headers: /content/drive/MyDrive/TRON/AI/generated


## 9. リポジトリ側への成果物コピー（必要時に実行）
通常の保存先はDriveです。このセルで `AI/dataset/metadata/09_integration_info/` に同じ一式をコピーできます。Cヘッダーはすでに `AI/generated/` に生成済みです。


In [9]:
REPO_META = AI_ROOT/'dataset/metadata/09_integration_info'
REPO_META.mkdir(parents=True, exist_ok=True)
for p in [VECTOR_CSV, OUT/'pc_reference_outputs.csv', run_path] + [OUT/(k+'.npy') for k in arrays]:
    dest = REPO_META/p.name
    if p.resolve() != dest.resolve():
        if dest.exists():
            assert sha256_file(dest) == sha256_file(p), f'Existing artifact differs: {dest}'
        shutil.copy2(p, dest)
    assert sha256_file(dest) == sha256_file(p)
print('Repository artifacts:', REPO_META)

Repository artifacts: /content/drive/MyDrive/TRON/AI/dataset/metadata/09_integration_info


## 次工程
同じ `pcm_int16.npy` の各行（16000 samples）をSTM32へ渡し、段階ごとの特徴量・INT8テンソル・予測クラスを比較します。
PC referenceとの許容差、ST Edge AI変換、メモリ量、実機推論時間はAI-09後半で確定します。
NPYにはヘッダーがあるため、そのままraw PCMとして読み込まないでください。配列を取り出して転送します。
Mel係数はCの行優先 `[fft_bin][mel_bin]`、CNN入力はNHWCです。C `roundf` とNumPyの丸め規則は異なるため、生成ヘッダーの契約に合わせて実装してください。
